# 05 — Export Models & Embeddings

Prepare trained models for deployment in the FastAPI sidecar:
1. Export semantic model checkpoint
2. Re-embed all anime with the fine-tuned model → 384-dim vectors
3. Export CF model checkpoint + anime index
4. Package everything into the `ml-models/` directory

**Requires**: `03_semantic_training.ipynb` and `04_cf_training.ipynb` completed.

In [ ]:
!pip install -q sentence-transformers torch scipy tqdm

In [ ]:
import json
import shutil
import numpy as np
import torch
from tqdm.auto import tqdm
from pathlib import Path
from sentence_transformers import SentenceTransformer

DATA_DIR = Path("data")
MODEL_DIR = Path("models")
# Prefer project-root ml-models/ when running from notebooks/ locally.
EXPORT_DIR = Path("../ml-models") if Path("../docker-compose.yml").exists() else Path("ml-models")
EXPORT_DIR.mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Export Semantic Model

In [ ]:
# Copy the fine-tuned sentence transformer to export directory
semantic_src = MODEL_DIR / "anime_semantic"
semantic_dst = EXPORT_DIR / "semantic"

if semantic_dst.exists():
    shutil.rmtree(semantic_dst)
shutil.copytree(semantic_src, semantic_dst)

# Verify it loads
model = SentenceTransformer(str(semantic_dst), device=str(device))
test_embed = model.encode(["test sentence"])
print(f"Semantic model exported: {semantic_dst}")
print(f"  Embedding dim: {test_embed.shape[1]}")

# Calculate model size
total_size = sum(f.stat().st_size for f in semantic_dst.rglob("*") if f.is_file())
print(f"  Total size: {total_size / 1e6:.1f} MB")

## 2. Re-Embed All Anime

Generate 384-dim embeddings for every anime using the fine-tuned model.
These will be loaded into pgvector for similarity search.

In [ ]:
# Load corpus (we embed the full text documents, not just titles)
corpus = []
with open(DATA_DIR / "corpus.jsonl", "r") as f:
    for line in f:
        corpus.append(json.loads(line))

print(f"Anime to embed: {len(corpus):,}")

In [ ]:
# Batch embed all anime
BATCH_SIZE = 64
texts = [entry["text"] for entry in corpus]

print("Embedding all anime...")
all_embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True  # L2 normalize for cosine similarity
)

print(f"Embeddings shape: {all_embeddings.shape}")

In [ ]:
# Save embeddings with AniList IDs for database import
embeddings_output = []
for i, entry in tqdm(enumerate(corpus), total=len(corpus), desc="Building embedding export"):
    embeddings_output.append({
        "anilist_id": entry["anilist_id"],
        "mal_id": entry.get("mal_id"),
        "title": entry["title"],
        "embedding": all_embeddings[i].tolist()
    })

with open(EXPORT_DIR / "anime_embeddings.jsonl", "w") as f:
    for entry in tqdm(embeddings_output, desc="Writing anime_embeddings.jsonl"):
        f.write(json.dumps(entry) + "\n")

print(f"Saved {len(embeddings_output):,} anime embeddings")
print(f"  File: {EXPORT_DIR / 'anime_embeddings.jsonl'}")
file_size = (EXPORT_DIR / 'anime_embeddings.jsonl').stat().st_size / 1e6
print(f"  Size: {file_size:.1f} MB")

## 3. Export CF Model

In [ ]:
# Copy CF checkpoint
cf_src = MODEL_DIR / "anime_cf" / "best_model.pt"
cf_dst = EXPORT_DIR / "cf"
cf_dst.mkdir(exist_ok=True)

shutil.copy2(cf_src, cf_dst / "model.pt")

# Copy anime index mapping
shutil.copy2(DATA_DIR / "cf_anime_index.json", cf_dst / "anime_index.json")

# Copy ID mapping
shutil.copy2(DATA_DIR / "id_map.json", EXPORT_DIR / "id_map.json")

cf_size = (cf_dst / 'model.pt').stat().st_size / 1e6
print(f"CF model exported: {cf_dst}")
print(f"  Checkpoint size: {cf_size:.1f} MB")
print(f"  Anime index: {cf_dst / 'anime_index.json'}")
print(f"  ID mapping: {EXPORT_DIR / 'id_map.json'}")

## 4. Export Summary

In [ ]:
print("=" * 50)
print("EXPORT SUMMARY")
print("=" * 50)

print(f"\nExport directory: {EXPORT_DIR.resolve()}")
print(f"\nContents:")

total_export_size = 0
for f_path in tqdm(sorted(EXPORT_DIR.rglob("*")), desc="Summarizing export"):
    if f_path.is_file():
        size = f_path.stat().st_size
        total_export_size += size
        rel_path = f_path.relative_to(EXPORT_DIR)
        print(f"  {rel_path}: {size / 1e6:.1f} MB")

print(f"\nTotal export size: {total_export_size / 1e6:.1f} MB")

print(f"\n[Deployment Instructions]")
print(f"  1. Download this 'ml-models/' directory")
print(f"  2. Place it at the project root: animetracker/ml-models/")
print(f"  3. Run: docker-compose up --build -d")
print(f"  4. The sidecar will load models from the Docker volume mount")
print(f"\n✓ Export complete!")